[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/ai-agents-certified/notebooks/day-02-state-and-reducers.ipynb#scrollTo=11223344)

---
# Day 2 · State Schemas and Reducers — TypedDict, add_messages, and Custom Reducers
**certified-journeys / ai-agents-certified** · Day 2 · State Management

> **Goal for today:** Understand why list fields in LangGraph state need reducers, implement a custom deduplicating reducer, and build a graph that uses `MessagesState` to accumulate an LLM conversation over multiple turns.

In [ ]:
%pip install -q langgraph langchain-openai langchain-core

## Step 1 · The Reducer Problem — Why List Fields Break Without One

By default LangGraph **replaces** each state field with whatever the node returns. For scalar fields (strings, ints) that's fine. For list fields that should **accumulate** across nodes it's a silent data-loss bug.

| Without reducer | With reducer |
|----------------|--------------|
| Node A writes `["a"]` → state has `["a"]` | Node A writes `["a"]` → state has `["a"]` |
| Node B writes `["b"]` → state has `["b"]` ← **overwrites A!** | Node B writes `["b"]` → state has `["a", "b"]` ← **merged!** |

A **reducer** is a function `(existing_value, new_value) -> merged_value` that LangGraph calls instead of a plain assignment. You attach it to a state field with `Annotated[type, reducer_fn]`.

The most important built-in reducer is `add_messages` — it merges LangChain message lists while deduplicating by message `id`.

In [ ]:
# Demonstrate the overwrite bug first
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# State WITHOUT a reducer on the list field
class NaiveState(TypedDict):
    items: list   # ← no reducer

def node_a(state: NaiveState) -> dict:
    return {"items": ["item_from_A"]}

def node_b(state: NaiveState) -> dict:
    return {"items": ["item_from_B"]}

g = StateGraph(NaiveState)
g.add_node("A", node_a)
g.add_node("B", node_b)
g.add_edge(START, "A")
g.add_edge("A", "B")
g.add_edge("B", END)
naive_graph = g.compile()

result = naive_graph.invoke({"items": []})
print("NaiveState result:", result["items"])
# Expected ["item_from_A", "item_from_B"] but you'll see only ["item_from_B"]

### What just happened?

- Node A's `"item_from_A"` was **silently overwritten** by Node B. The final state only contains what the last node returned.
- **This is not a LangGraph bug** — it's intentional default behaviour. Replace semantics are correct for scalar fields. For accumulating fields you must opt in to a reducer.
- The fix is `Annotated[list, operator.add]` — a one-line change to the schema.

## Step 2 · Fixing the Bug with `Annotated` and a Built-in Reducer

Python's `typing.Annotated` lets you attach metadata to a type annotation. LangGraph reads that metadata to find the reducer function.

```python
Annotated[list, operator.add]  # merges lists by concatenation
```

The reducer function signature is always `(current_value, update_value) -> merged_value`.

In [ ]:
import operator
from typing import Annotated

# State WITH a reducer on the list field
class ReducedState(TypedDict):
    # operator.add for lists = concatenation
    items: Annotated[list, operator.add]

def node_a_v2(state: ReducedState) -> dict:
    return {"items": ["item_from_A"]}

def node_b_v2(state: ReducedState) -> dict:
    return {"items": ["item_from_B"]}

g2 = StateGraph(ReducedState)
g2.add_node("A", node_a_v2)
g2.add_node("B", node_b_v2)
g2.add_edge(START, "A")
g2.add_edge("A", "B")
g2.add_edge("B", END)
reduced_graph = g2.compile()

result2 = reduced_graph.invoke({"items": []})
print("ReducedState result:", result2["items"])
# Now correctly ["item_from_A", "item_from_B"]

### What just happened?

- **`Annotated[list, operator.add]`** tells LangGraph: when a node returns a new `items` value, don't replace — call `operator.add(current, update)` to merge.
- `operator.add` on two lists is identical to `current + update` — pure concatenation.
- **The schema change is the only change** — the node functions are unchanged. This is by design: nodes are unaware of reducers; they just return their local contribution.

## Step 3 · Implementing a Custom Deduplicating Reducer

Sometimes concatenation isn't right. Suppose multiple nodes might add the same tag — you want a **set-union** merge. You can write any function with the `(current, update) -> merged` signature.

In [ ]:
from typing import Annotated

# Custom reducer: merge two lists, keeping insertion order, removing duplicates
def deduplicate_reducer(current: list, update: list) -> list:
    """
    Merges `update` into `current`, preserving existing order and
    skipping any items already present in `current`.
    """
    existing = set(current)            # O(1) membership test
    new_items = [x for x in update if x not in existing]
    return current + new_items

# State using the custom reducer
class TagState(TypedDict):
    tags: Annotated[list, deduplicate_reducer]
    content: str

def tagger_a(state: TagState) -> dict:
    return {"tags": ["python", "ai", "langgraph"]}

def tagger_b(state: TagState) -> dict:
    # 'ai' already exists — should NOT appear twice
    return {"tags": ["ai", "agents", "llm"]}

def tagger_c(state: TagState) -> dict:
    # 'python' and 'llm' already exist — only 'tutorial' should be added
    return {"tags": ["python", "llm", "tutorial"]}

g3 = StateGraph(TagState)
g3.add_node("A", tagger_a)
g3.add_node("B", tagger_b)
g3.add_node("C", tagger_c)
g3.add_edge(START, "A")
g3.add_edge("A", "B")
g3.add_edge("B", "C")
g3.add_edge("C", END)
tag_graph = g3.compile()

result3 = tag_graph.invoke({"tags": [], "content": "sample content"})
print("Final tags (deduplicated):", result3["tags"])
print("Expected: ['python', 'ai', 'langgraph', 'agents', 'llm', 'tutorial']")

### What just happened?

- **`deduplicate_reducer`** implements the merge contract: receives `current` (what the state holds before this node) and `update` (what the node returned), returns the merged value.
- LangGraph calls this function transparently — nodes don't know it exists. They just return a list of tags.
- **Order is preserved** — we use a set only for membership testing, not for storage. This is important when downstream code depends on tag order.
- You can test a reducer in isolation with plain Python — no graph needed — because it's just a function.

## Step 4 · `add_messages` and `MessagesState`

`add_messages` is LangGraph's built-in reducer for LLM message lists. It does two things beyond plain concatenation:

1. **Deduplication by `id`** — if you return a message with the same `id` as an existing one, it replaces rather than duplicates. This supports message editing.
2. **Type coercion** — tuples like `("human", "text")` are automatically converted to `HumanMessage` objects.

`MessagesState` is a convenience subclass of `TypedDict` with a single `messages` field pre-wired to `add_messages`. Import it and you're done.

In [ ]:
from langgraph.graph.message import add_messages
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# Demonstrate add_messages directly — no graph needed
existing_messages = [
    SystemMessage(content="You are a helpful assistant.", id="sys-001"),
    HumanMessage(content="Hello!", id="human-001"),
]

new_message = AIMessage(content="Hi! How can I help you today?", id="ai-001")

merged = add_messages(existing_messages, [new_message])
print("After first merge:")
for m in merged:
    print(f"  [{m.id}] {m.__class__.__name__}: {m.content}")

# Now demonstrate dedup: updating an existing message by id
updated_ai = AIMessage(content="Hi! I'm ready to assist!", id="ai-001")  # same id
merged2 = add_messages(merged, [updated_ai])
print()
print("After update (same id):")
for m in merged2:
    print(f"  [{m.id}] {m.__class__.__name__}: {m.content}")
print(f"  → Total messages: {len(merged2)} (not 4 — deduplication worked)")

### What just happened?

- **`add_messages` is just a function** — you can call it directly outside a graph to understand its behaviour.
- When the second AIMessage with `id="ai-001"` was passed, `add_messages` replaced the first one rather than appending — the list length stayed at 3.
- **This makes multi-turn editing possible**: streaming LLMs can emit partial messages with the same id, and each chunk replaces the previous partial — you always have the latest version in state.

## Step 5 · Building a Multi-Turn Graph with `MessagesState`

Now we wire a real graph that simulates three LLM turns. We'll use a **mock LLM** so this cell runs without an API key.

In production you'd swap the mock for `ChatOpenAI(model="gpt-4o-mini")` — the graph structure is identical.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import MessagesState, StateGraph, START, END

# Mock LLM — returns canned responses based on turn count
# In production: replace with ChatOpenAI(model="gpt-4o-mini").invoke(messages)
MOCK_RESPONSES = [
    "Nice to meet you! What would you like to know?",
    "LangGraph models agents as directed graphs — nodes are functions, edges are transitions.",
    "Exactly right. State flows through every node so all functions share information.",
]

_turn_counter = {"count": 0}  # simple closure counter for mock

def call_model(state: MessagesState) -> dict:
    """Simulates an LLM call: reads all messages so far, returns an AI reply."""
    # In production this would be:
    # response = llm.invoke(state["messages"])
    # return {"messages": [response]}

    turn = _turn_counter["count"] % len(MOCK_RESPONSES)
    _turn_counter["count"] += 1
    response_text = MOCK_RESPONSES[turn]

    # Return a list — add_messages will merge it with the existing messages
    return {"messages": [AIMessage(content=response_text)]}

# Build a simple single-node conversation graph
convo_builder = StateGraph(MessagesState)
convo_builder.add_node("llm", call_model)
convo_builder.add_edge(START, "llm")
convo_builder.add_edge("llm", END)
convo_graph = convo_builder.compile()

# Simulate three turns by invoking the graph three times,
# carrying the accumulated messages forward each time
messages = []

user_turns = [
    "Hello! I'm learning LangGraph.",
    "Can you explain what LangGraph is in one sentence?",
    "So nodes communicate through shared state?",
]

for i, user_text in enumerate(user_turns):
    messages.append(HumanMessage(content=user_text))
    result = convo_graph.invoke({"messages": messages})
    messages = result["messages"]   # carry forward the full accumulated list
    print(f"Turn {i+1}:")
    print(f"  Human : {user_text}")
    print(f"  AI    : {messages[-1].content}")
    print()

print(f"Total messages in state: {len(messages)}")

### What just happened?

- **`MessagesState` provides `messages: Annotated[list, add_messages]` out of the box** — you don't write any schema boilerplate for conversation history.
- After three turns, the state holds **6 messages** (3 human + 3 AI). Each turn appended to the existing list rather than overwriting it — that's the reducer at work.
- **In production** the graph would read `state["messages"]` to give the LLM the full conversation context, enabling multi-turn coherent conversations.

## Step 6 · TypedDict vs Pydantic BaseModel State

LangGraph accepts both `TypedDict` and Pydantic `BaseModel` as state schemas. The behaviour is identical — choose based on your project's conventions.

| Feature | TypedDict | Pydantic BaseModel |
|---------|-----------|--------------------|
| Runtime validation | No | Yes — raises on bad types |
| Default values | No (use `total=False`) | Yes — `Field(default=...)` |
| IDE autocomplete | Good | Excellent |
| JSON schema | Manual | Auto-generated |
| Performance overhead | Zero | Small (~5-10 µs/validation) |

For most LangGraph work, TypedDict is sufficient. Use Pydantic when you need input validation (e.g., the graph is called from an API endpoint).

In [ ]:
import operator
from typing import Annotated
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END

# --- Pydantic version of ReducedState ---
class PydanticState(BaseModel):
    # Annotated works identically in Pydantic models
    items: Annotated[list, operator.add] = Field(default_factory=list)
    label: str = "default"

def node_x(state: PydanticState) -> dict:
    return {"items": ["from_X"], "label": "set_by_X"}

def node_y(state: PydanticState) -> dict:
    return {"items": ["from_Y"]}  # partial update — label unchanged

pg = StateGraph(PydanticState)
pg.add_node("X", node_x)
pg.add_node("Y", node_y)
pg.add_edge(START, "X")
pg.add_edge("X", "Y")
pg.add_edge("Y", END)
pydantic_graph = pg.compile()

# Both TypedDict and Pydantic accept a plain dict as invoke() input
result_pydantic = pydantic_graph.invoke({"items": [], "label": "initial"})
print("Pydantic graph result:")
print("  items :", result_pydantic["items"])
print("  label :", result_pydantic["label"])

# Compare to TypedDict version (from Step 2)
result_typed = reduced_graph.invoke({"items": []})
print()
print("TypedDict graph items:", result_typed["items"])
print("Behaviour identical:", result_pydantic["items"] == result_typed["items"])

### What just happened?

- **Both schemas produce the same result** — the reducer (`operator.add`) works identically regardless of whether the schema is TypedDict or Pydantic.
- **Partial updates work in Pydantic too** — `node_y` only returned `items`; the `label` field retained the value set by `node_x`.
- **Pydantic's `default_factory=list`** means you can invoke the graph with an empty dict: `pydantic_graph.invoke({})` — the `items` field defaults to `[]` automatically.

In [ ]:
# Challenge: Implement a reducer that tracks the running maximum
#
# 1. Write a reducer function `keep_max(current: int, update: int) -> int`
#    that returns whichever value is larger.
#
# 2. Create a state schema `ScoreState` with two fields:
#    - `high_score: Annotated[int, keep_max]`
#    - `scores_seen: Annotated[list, operator.add]`
#
# 3. Build a 4-node graph where each node returns a different score value
#    (e.g., 72, 88, 95, 61) and a list containing that score.
#    Wire them in sequence: A → B → C → D → END.
#
# 4. Invoke the graph with {"high_score": 0, "scores_seen": []}
#    and verify:
#    - high_score = 95 (the maximum across all four nodes)
#    - scores_seen = [72, 88, 95, 61] (all scores accumulated in order)

# TODO: def keep_max(current: int, update: int) -> int:
# TODO: class ScoreState(TypedDict): ...
# TODO: define four node functions
# TODO: build, compile, invoke, and print results

---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Default merge behaviour | LangGraph **replaces** each field — correct for scalars, wrong for lists |
| `Annotated[T, fn]` | Attaches a reducer function to a state field |
| Reducer signature | `(current_value, update_value) -> merged_value` — called transparently |
| `operator.add` | Built-in list concatenation reducer — the simplest choice |
| `add_messages` | Merges message lists and deduplicates by message `id` |
| `MessagesState` | TypedDict subclass with `messages: Annotated[list, add_messages]` pre-wired |
| Pydantic compatibility | `BaseModel` works anywhere `TypedDict` does — reducers behave identically |

> **Tip:** Every state field that uses a mutable default needs a reducer. If you skip the reducer, later nodes silently overwrite what earlier nodes wrote — a bug that only shows up at scale.

---
## What's next
**Day 3** → Conditional Edges and Routing Logic — inspect message content to decide which node runs next, implement multi-branch routing, and add a fallback for unknown intents.

Mark Day 2 complete in your [tracker](../index.html).